# Q0.7 Weight Failure Map

This notebook maps where **firmware-normalised `Q0.7` weights** are sufficient for the Trouper combiner and where they still fail.

We assume the live combiner is reinterpreted as:

$$y = \mathrm{sat8}\left(\left(\sum_k W_k x_kight) \gg 7 \gg 1ight)$$

where `W_k` is a signed 8-bit complex weight and therefore represents an effective weight of `W_k / 128`.

The firmware model used here is optimistic but realistic:
- compute the matched-filter direction `conj(h)`
- choose the **best scalar normalisation** for that direction
- quantise into `Q0.7`
- among all scalings that avoid combiner clipping, keep the one with the best output SNR

This tells us whether `Q0.7` is workable with smart firmware, not just whether `127` is too hot.


In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

RNG = np.random.default_rng(1)
NR = 4
PLOT_DIR = pathlib.Path('../plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)
Q_SCALE = 128.0   # Q0.7
FULL_SCALE = 127
N_TRIALS = 120


---
## 1  Model

For a complex branch vector $h$, the ideal matched-combining direction is $d = h^* / \max |h|$.
Firmware chooses a real scalar $c$ and writes:

$$W = \mathrm{round}(128 \cdot c \cdot d)$$

The combiner interprets this as effective weight $W/128$. We search over `c` and keep the best no-clipping result.


In [ ]:
def coherent_profile(strong_amp, spread_db, nr=NR):
    return strong_amp * 10.0 ** (-np.linspace(0.0, spread_db, nr) / 20.0)


def matched_direction(h):
    d = np.conj(h)
    peak = np.max(np.abs(d))
    return d / peak if peak > 0 else d


def quantise_q07(direction, c):
    q = np.round(Q_SCALE * c * direction.real).astype(int) + 1j * np.round(Q_SCALE * c * direction.imag).astype(int)
    q = np.clip(q.real, -128, 127).astype(int) + 1j * np.clip(q.imag, -128, 127).astype(int)
    return q


def combiner_q07_metrics(h, q):
    acc = np.sum(q * h)
    y_pre = acc / Q_SCALE / 2.0
    clipped = (abs(y_pre.real) > FULL_SCALE) or (abs(y_pre.imag) > FULL_SCALE)
    signal = abs(np.sum(q * h)) ** 2
    noise = np.sum(np.abs(q) ** 2)
    snr = signal / noise if noise > 0 else 0.0
    return y_pre, clipped, snr


def best_q07_scale(h, c_grid):
    d = matched_direction(h)
    ideal_snr = np.sum(np.abs(h) ** 2)
    best = None
    for c in c_grid:
        q = quantise_q07(d, c)
        if np.all(q == 0):
            continue
        y_pre, clipped, snr = combiner_q07_metrics(h, q)
        if clipped:
            continue
        cand = {
            'c': c,
            'q': q,
            'snr': snr,
            'ideal_snr': ideal_snr,
            'loss_db': 10.0 * np.log10(ideal_snr / snr) if snr > 0 else np.inf,
            'branches_used': int(np.count_nonzero(np.abs(q) > 0)),
            'peak_pre': max(abs(y_pre.real), abs(y_pre.imag)),
        }
        if best is None or cand['snr'] > best['snr']:
            best = cand
    return best


---
## 2  Coherent Equal-Phase Sweep

This is the worst clipping case because all branches add constructively.


In [ ]:
amps = np.linspace(8, 100, 47)
spreads = np.linspace(0, 20, 41)
c_grid = np.linspace(0.01, 1.0, 240)

loss_map = np.full((len(amps), len(spreads)), np.nan)
used_map = np.full((len(amps), len(spreads)), np.nan)
peak_map = np.full((len(amps), len(spreads)), np.nan)
feasible_map = np.zeros((len(amps), len(spreads)), dtype=bool)

for ia, A in enumerate(amps):
    for isd, spread in enumerate(spreads):
        branch_amps = coherent_profile(A, spread)
        h = branch_amps.astype(complex)  # equal phase worst-case
        best = best_q07_scale(h, c_grid)
        if best is not None:
            feasible_map[ia, isd] = True
            loss_map[ia, isd] = best['loss_db']
            used_map[ia, isd] = best['branches_used']
            peak_map[ia, isd] = best['peak_pre']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True, sharey=True)

im0 = axes[0].contourf(spreads, amps, np.where(feasible_map, 1.0, 0.0), levels=[-0.1, 0.1, 0.9, 1.1], cmap='RdYlGn')
axes[0].set_title('Feasible no-clip Q0.7 solution')
axes[0].set_xlabel('SNR spread d (dB)')
axes[0].set_ylabel('Strongest branch amplitude A (counts)')

im1 = axes[1].contourf(spreads, amps, np.nan_to_num(loss_map, nan=10.0), levels=np.linspace(0, 6, 13), cmap='viridis')
axes[1].set_title('Best no-clip SNR loss (dB)')
axes[1].set_xlabel('SNR spread d (dB)')

im2 = axes[2].contourf(spreads, amps, np.nan_to_num(used_map, nan=0.0), levels=np.arange(-0.5, 5.5, 1), cmap='plasma')
axes[2].set_title('Branches with nonzero quantised weight')
axes[2].set_xlabel('SNR spread d (dB)')

for ax in axes:
    ax.grid(True, alpha=0.2)

fig.colorbar(im1, ax=axes[1], label='dB')
fig.colorbar(im2, ax=axes[2], label='count')
fig.tight_layout()
plt.savefig(PLOT_DIR / 'q07_failure_map_coherent.png', bbox_inches='tight')
plt.show()
print('Saved: sim/plots/q07_failure_map_coherent.png')


---
## 3  Random-Phase Monte Carlo

The equal-phase case is conservative. Here we sample random channel phases and ask how often firmware-normalised `Q0.7` still fails to find a no-clip solution, and what loss remains when it succeeds.


In [ ]:
mc_amps = [90, 64, 45, 32]
mc_spreads = np.linspace(0, 20, 11)
fail_prob = {A: [] for A in mc_amps}
p95_loss = {A: [] for A in mc_amps}
mean_loss = {A: [] for A in mc_amps}

for A in mc_amps:
    for spread in mc_spreads:
        fails = 0
        losses = []
        for _ in range(N_TRIALS):
            branch_amps = coherent_profile(A, spread)
            phases = RNG.uniform(0, 2*np.pi, NR)
            h = branch_amps * np.exp(1j * phases)
            best = best_q07_scale(h, c_grid)
            if best is None:
                fails += 1
            else:
                losses.append(best['loss_db'])
        fail_prob[A].append(fails / N_TRIALS)
        mean_loss[A].append(np.mean(losses) if losses else np.nan)
        p95_loss[A].append(np.percentile(losses, 95) if losses else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
cols = ['tab:red', 'tab:orange', 'tab:blue', 'tab:green']
for A, col in zip(mc_amps, cols):
    axes[0].plot(mc_spreads, fail_prob[A], color=col, lw=2, label=f'A={A}')
    axes[1].plot(mc_spreads, p95_loss[A], color=col, lw=2, label=f'A={A}')

axes[0].set_title('Probability no no-clip Q0.7 solution exists')
axes[0].set_xlabel('SNR spread d (dB)')
axes[0].set_ylabel('Failure probability')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].set_title('95th-percentile SNR loss when feasible')
axes[1].set_xlabel('SNR spread d (dB)')
axes[1].set_ylabel('Loss (dB)')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.savefig(PLOT_DIR / 'q07_failure_map_random_phase.png', bbox_inches='tight')
plt.show()
print('Saved: sim/plots/q07_failure_map_random_phase.png')


---
## 4  Corner Table


In [ ]:
corners = [
    ('equal_90', 90, 0, np.zeros(NR)),
    ('equal_64', 64, 0, np.zeros(NR)),
    ('equal_45', 45, 0, np.zeros(NR)),
    ('mild_90', 90, 6, np.zeros(NR)),
    ('moderate_90', 90, 12, np.zeros(NR)),
]

print(f"{'case':12} {'best c':>8} {'weights':>28} {'used':>6} {'loss(dB)':>10} {'peak_pre':>10}")
print('-' * 88)
for name, A, spread, phases in corners:
    h = coherent_profile(A, spread) * np.exp(1j * phases)
    best = best_q07_scale(h, c_grid)
    if best is None:
        print(f"{name:12} {'NONE':>8} {'-':>28} {'-':>6} {'-':>10} {'-':>10}")
    else:
        w = '[' + ', '.join(f'{int(z.real):+d}{int(z.imag):+d}j' for z in best['q']) + ']'
        print(f"{name:12} {best['c']:8.3f} {w:>28} {best['branches_used']:6d} {best['loss_db']:10.4f} {best['peak_pre']:10.2f}")


---
## 5  Conclusions

Read the plots as follows:
- **Failure probability** means firmware could not find any nonzero `Q0.7` weight vector in the matched-filter direction that stayed below the combiner clipping limit.
- **Loss** means the best available no-clip `Q0.7` vector still lost combining gain because quantisation forced some branches toward zero or distorted the ratios.

The equal-phase map is the hard upper bound. Random phases show how often that bound is actually encountered.
